# Stage 1.5: Create Seasonal Subsets

This notebook splits the processed `merged_gdf.feather` into **8 subset datasets**,
one for each combination of:
- **Season**: Spring, Summer, Autumn, Winter
- **Depth layer**: Surface (`saisuipoin == 1`), Bottom (`saisuipoin == 2`)

For each subset it computes the **yearly mean DO per station** and saves a
`yearly_do_df.csv` inside:
```
data/processed/seasonal/{season_lower}_{layer_lower}/
```

> **Note**: The station-to-station distance matrix (`dist_matrix_km.csv`) depends
> only on station geography (not on season or layer) and is therefore **shared**.
> It is produced once by `08_Decorrelation_Analysis_v3.ipynb` and lives at
> `data/processed/dist_matrix_km.csv`.

In [1]:
import pandas as pd
import sys
from pathlib import Path

# Make project scripts importable
project_root = Path("../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from scripts.paths import ProjectPaths
from scripts.data_loader import load_and_process_data

print("Project root:", project_root)

Project root: /home/reyha


In [2]:
# Parameters (injectable via papermill / run_pipeline_step.py)
min_periods = 15  # stations must have >= this many years of data to be included
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
layers  = {'surface': 1, 'bottom': 2}

In [3]:
# Parameters
min_periods = 15


## 1. Load the merged dataset

In [4]:
# Load the cleaned, merged dataset produced by 01_Data_Processing.ipynb
merged_gdf = load_and_process_data(force_reload=False)

print(f"Loaded merged_gdf: {merged_gdf.shape}")
print(f"Seasons present : {sorted(merged_gdf['season'].unique())}")
print(f"Layers present  : {sorted(merged_gdf['saisuipoin'].unique())}")

Loading processed data from /home/reyha/SIS_Analysis_WSL/data/processed/merged_gdf.feather...


Loaded merged_gdf: (42326, 65)
Seasons present : ['Autumn', 'Spring', 'Summer', 'Winter']
Layers present  : [np.int64(1), np.int64(2)]


## 2. Create one subset directory per season × layer combination

For each subset:
1. Filter rows by `season` and `saisuipoin`.
2. Drop rows where `do_` is NaN.
3. Aggregate to yearly mean DO per station (`zettaicode`).
4. Drop stations with fewer than `min_periods` non-NaN years.
5. Save as `yearly_do_df.csv` in the appropriate subdirectory.

In [5]:
results_summary = []

for season in seasons:
    for layer_name, layer_code in layers.items():

        # --- 1. Filter ---
        subset = merged_gdf[
            (merged_gdf['season'] == season) &
            (merged_gdf['saisuipoin'] == layer_code)
        ].dropna(subset=['do_']).copy()

        subset['do_'] = pd.to_numeric(subset['do_'], errors='coerce')

        # --- 2. Yearly mean DO per station ---
        yearly_do = (
            subset
            .groupby(['zettaicode', 'year'])['do_']
            .mean()
            .unstack('year')
        )

        # --- 3. Drop stations below min_periods threshold ---
        yearly_do = yearly_do[yearly_do.notna().sum(axis=1) >= min_periods]

        # Ensure the station index is zero-padded 8-digit strings
        yearly_do.index = yearly_do.index.astype(str).str.zfill(8)

        # --- 4. Save ---
        out_dir = ProjectPaths.ensure_subset_dir(season, layer_name)
        out_path = out_dir / 'yearly_do_df.csv'
        yearly_do.to_csv(out_path, index=True)

        results_summary.append({
            'season': season,
            'layer': layer_name,
            'saisuipoin': layer_code,
            'n_stations': yearly_do.shape[0],
            'n_years': yearly_do.shape[1],
            'output': str(out_path)
        })
        print(f"[{season:6s} / {layer_name:7s}]  stations={yearly_do.shape[0]:3d}  years={yearly_do.shape[1]}  -> {out_path}")

print("\nAll subsets saved.")

[Spring / surface]  stations=124  years=43  -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spring_surface/yearly_do_df.csv
[Spring / bottom ]  stations=124  years=43  -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spring_bottom/yearly_do_df.csv
[Summer / surface]  stations=124  years=43  -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/summer_surface/yearly_do_df.csv
[Summer / bottom ]  stations=124  years=43  -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/summer_bottom/yearly_do_df.csv


[Autumn / surface]  stations=124  years=43  -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/autumn_surface/yearly_do_df.csv


[Autumn / bottom ]  stations=124  years=43  -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/autumn_bottom/yearly_do_df.csv
[Winter / surface]  stations=124  years=43  -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/winter_surface/yearly_do_df.csv
[Winter / bottom ]  stations=124  years=43  -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/winter_bottom/yearly_do_df.csv

All subsets saved.


## 3. Summary

In [6]:
summary_df = pd.DataFrame(results_summary)
display(summary_df)

,season,layer,saisuipoin,n_stations,n_years,output
0,Spring,surface,1,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
1,Spring,bottom,2,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
2,Summer,surface,1,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
3,Summer,bottom,2,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
4,Autumn,surface,1,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
5,Autumn,bottom,2,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
6,Winter,surface,1,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
7,Winter,bottom,2,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
